# Handle board


In [ ]:
import zipfile

zip_path = "/content/raw_boards.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
import cv2
import numpy as np

def extract_and_slice_board(image_path, rows=9, cols=16, model_input_size=(48, 48), gap=1):
    # load image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read image from {image_path}")

    original_img = img.copy()
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blur, 50, 150)

    # Find small tiles to determine the main board
    contours, _ = cv2.findContours(edges, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    img_h, img_w, _ = img.shape
    estimated_tile_w = img_w // cols
    estimated_tile_h = img_h // rows
    estimated_tile_area = estimated_tile_w * estimated_tile_h

    min_area = estimated_tile_area * 0.1
    max_area = estimated_tile_area * 3.0

    tile_contours = []
    for c in contours:
        x_c, y_c, w_c, h_c = cv2.boundingRect(c)
        area_c = w_c * h_c
        if area_c < min_area or area_c > max_area:
            continue

        aspect_ratio = w_c / h_c if w_c > h_c else h_c / w_c
        if aspect_ratio > 1.5:
            continue

        tile_contours.append(c)

    if len(tile_contours) < 50:
        raise ValueError("Not enough tiles found to identify the board.")

    # Create bounding box for the entire board (tight fit - padding = 0)
    x_min, y_min = img_w, img_h
    x_max, y_max = 0, 0

    for c in tile_contours:
        x_c, y_c, w_c, h_c = cv2.boundingRect(c)
        if x_c < x_min: x_min = x_c
        if y_c < y_min: y_min = y_c
        if (x_c + w_c) > x_max: x_max = (x_c + w_c)
        if (y_c + h_c) > y_max: y_max = (y_c + h_c)

    # Set padding to 0 as requested
    padding = 0
    x_final = max(0, x_min - padding)
    y_final = max(0, y_min - padding)
    w_final = min(img_w - x_final, (x_max - x_min) + padding * 2)
    h_final = min(img_h - y_final, (y_max - y_min) + padding * 2)

    board_img = original_img[y_final:y_final+h_final, x_final:x_final+w_final]

    # SLICE BOARD ACCURATELY WITH GAP BETWEEN TILES
    # Calculate standard tile size (float) minus the gaps
    tile_w_float = (w_final - (cols - 1) * gap) / cols
    tile_h_float = (h_final - (rows - 1) * gap) / rows

    tiles = []

    for r in range(rows):
        for c in range(cols):
            # Start coordinates: accumulate width of previous tiles and previous gaps
            start_x = int(round(c * (tile_w_float + gap)))
            start_y = int(round(r * (tile_h_float + gap)))

            # End coordinates
            end_x = int(round(start_x + tile_w_float))
            end_y = int(round(start_y + tile_h_float))

            # Crop tile
            tile = board_img[start_y:end_y, start_x:end_x]

            if tile.shape[0] > 0 and tile.shape[1] > 0:
                tile_resized = cv2.resize(tile, model_input_size)
                tiles.append(tile_resized)
            else:
                print(f"warning: row {r}, col {c} empty, skip.")
    return board_img, tiles

# Test script
if __name__ == "__main__":
    try:
        # Use gap=2 as per test configuration
        board, tile_list = extract_and_slice_board("/content/raw_boards/4.png", gap=2)

        cv2.imwrite("cropped_board.png", board)
        if len(tile_list) >= 144:
            # Save the first tile (top-left) and the 20th tile to check edges
            cv2.imwrite("tile_0.png", tile_list[0])
            cv2.imwrite("tile_19.png", tile_list[19])

    except Exception as e:
        print(f"Err: {e}")

In [ ]:
import os
import glob

def process_batch_images(input_dir, output_dir, gap=1):
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created: {output_dir}")

    # Scan for image files in input directory
    image_paths = []
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        image_paths.extend(glob.glob(os.path.join(input_dir, ext)))

    if not image_paths:
        print(f"No images found in '{input_dir}'!")
        return

    print(f"Found {len(image_paths)} images\n")
    total_extracted = 0

    for idx, img_path in enumerate(image_paths):
        filename = os.path.basename(img_path)
        print(f"[{idx + 1}/{len(image_paths)}] Processing: {filename}")

        try:
            # Slice image into tiles
            _, tiles = extract_and_slice_board(img_path, gap=gap)

            # Save each small tile
            for i, tile in enumerate(tiles):
                # Format: board_01_tile_001.png
                out_name = f"board_{idx+1:02d}_tile_{i:03d}.png"
                out_path = os.path.join(output_dir, out_name)

                cv2.imwrite(out_path, tile)
                total_extracted += 1

            print(f"Saved {len(tiles)} tiles.")

        except Exception as e:
            print(f"Error at {filename}: {e}")

    print(f"\nTotal {total_extracted} small images saved in: '{output_dir}'.")

# Execution script
if __name__ == "__main__":
    # Source directory for board images
    INPUT_FOLDER = "raw_boards"

    # Target directory for extracted tiles
    OUTPUT_FOLDER = "dataset_unlabeled"

    # Execute with gap=1
    process_batch_images(INPUT_FOLDER, OUTPUT_FOLDER, gap=1)

Found 10 img

[1/10] Cutting: 10.png
Save 144 tiles.
[2/10] Cutting: 5.png
Save 144 tiles.
[3/10] Cutting: 3.png
Save 144 tiles.
[4/10] Cutting: 1.png
Save 144 tiles.
[5/10] Cutting: 9.png
Save 144 tiles.
[6/10] Cutting: 2.png
Save 144 tiles.
[7/10] Cutting: 6.png
Save 144 tiles.
[8/10] Cutting: 7.png
Save 144 tiles.
[9/10] Cutting: 8.png
Save 144 tiles.
[10/10] Cutting: 4.png
Save 144 tiles.

Total 1440 small img in: 'dataset_unlabeled'.


In [ ]:
!rm -rf dataset_clustered/class_{56..99}

In [ ]:
import cv2
import numpy as np
import os
import glob
import shutil

def calculate_mse(img1, img2):
    err = np.sum((img1.astype("float") - img2.astype("float")) ** 2)
    err /= float(img1.shape[0] * img1.shape[1])
    return err

def auto_cluster_images(input_dir, output_dir, threshold=1500):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    image_paths = glob.glob(os.path.join(input_dir, "*.png"))
    print(f"Starting clustering for {len(image_paths)} images...")

    # Store representative templates for each class
    templates = []

    for img_path in image_paths:
        img = cv2.imread(img_path)
        if img is None:
            continue

        matched_class = -1

        # Compare current image with existing templates
        for idx, template in enumerate(templates):
            error = calculate_mse(img, template)
            if error < threshold:
                matched_class = idx
                break

        # If no match found, create a new class
        if matched_class == -1:
            matched_class = len(templates)
            templates.append(img)

            # Create directory for the new class
            class_dir = os.path.join(output_dir, f"class_{matched_class:02d}")
            os.makedirs(class_dir, exist_ok=True)

        # Copy image to the corresponding class folder
        dest_path = os.path.join(output_dir, f"class_{matched_class:02d}", os.path.basename(img_path))
        shutil.copy(img_path, dest_path)

    print(f"Finished! Automatically clustered into {len(templates)} groups.")

if __name__ == "__main__":
    # Input: Folder containing cut images
    INPUT = "dataset_unlabeled"

    # Output: Folder for clustered images
    OUTPUT = "dataset_clustered"

    auto_cluster_images(INPUT, OUTPUT, threshold=5000)

Bắt đầu phân cụm 1440 ảnh...
Hoàn tất! Đã tự động chia thành 139 nhóm.


In [ ]:
import glob

def remove_exact_duplicates(folder_path):
    image_paths = sorted(glob.glob(os.path.join(folder_path, "*.png")))
    deleted_count = 0

    unique_images = []

    for path in image_paths:
        img = cv2.imread(path)
        if img is None:
            continue

        is_duplicate = False

        for unique_img in unique_images:
            if calculate_mse(img, unique_img) == 0:
                is_duplicate = True
                break

        if is_duplicate:
            os.remove(path)
            deleted_count += 1
        else:
            unique_images.append(img)

    print(f"Done! Removed {deleted_count} exact duplicates in {folder_path}.")

In [ ]:
for i in range(166):
  if i < 10:
    folder_path = f"/content/dataset_clustered/class_0{i}"
  else:
    folder_path = f"/content/dataset_clustered/class_{i}"
  remove_exact_duplicates(folder_path)

In [ ]:
import shutil

# shutil.make_archive("output_filename", "format", "directory_to_compress")
shutil.make_archive("dataset_backup", "zip", "/content/dataset_clustered")

'/content/dataset_backup.zip'

# Load train

In [1]:
import zipfile

zip_path = "/content/dataset_clustered.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

In [2]:
import torch
from torch import nn
from torch import optim
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [ ]:
data_transforms = transforms.Compose([
    transforms.Resize((48, 48)),            # make sure img size is 48x48
    transforms.RandomRotation(degrees=5),  
    transforms.ColorJitter(
        brightness=0.1,         
        contrast=0.1     
    ),
    transforms.ToTensor(),     
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5], 
        std=[0.5, 0.5, 0.5]               
    )
])

In [4]:
DATA_DIR = "/content/dataset_clustered"

full_dataset = torchvision.datasets.ImageFolder(
    root=DATA_DIR,
    transform=data_transforms
)

In [5]:
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

In [6]:
BATCH_SIZE = 64
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

In [ ]:
class PikachuCNN(nn.Module):
    def __init__(self, num_classes=36):
        super(PikachuCNN, self).__init__()

        # (3, 48, 48) -> (32, 24, 24)
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # (32, 24, 24) -> (64, 12, 12)
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # (64, 12, 12) -> (128, 6, 6)
        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Fully Connected
        # 128 channels * 6 width * 6 height = 4608
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc_layers(x)
        return x

model = PikachuCNN(num_classes=36).to(device)

In [9]:
import time
import copy

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=25):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        start_time = time.time()
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)


        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader

            running_loss = 0.0
            running_corrects = 0


            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        time_elapsed = time.time() - start_time
        print(f'running time epoch: {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s\n')

    print(f'best Val Accuracy: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model

In [14]:
EPOCHS = 14
LEARNING_RATE = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [15]:
best_model = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=EPOCHS)

torch.save(best_model.state_dict(), 'pikachu_model_best.pth')

Epoch 1/14
----------
Train Loss: 0.1787 Acc: 0.9535
Val Loss: 0.0366 Acc: 1.0000
running time epoch: 0m 2s

Epoch 2/14
----------
Train Loss: 0.1514 Acc: 0.9628
Val Loss: 0.0723 Acc: 1.0000
running time epoch: 0m 2s

Epoch 3/14
----------
Train Loss: 0.1448 Acc: 0.9628
Val Loss: 0.0624 Acc: 0.9815
running time epoch: 0m 2s

Epoch 4/14
----------
Train Loss: 0.0805 Acc: 0.9814
Val Loss: 0.0411 Acc: 0.9815
running time epoch: 0m 3s

Epoch 5/14
----------
Train Loss: 0.0502 Acc: 0.9953
Val Loss: 0.0155 Acc: 1.0000
running time epoch: 0m 2s

Epoch 6/14
----------
Train Loss: 0.0349 Acc: 1.0000
Val Loss: 0.0119 Acc: 1.0000
running time epoch: 0m 2s

Epoch 7/14
----------
Train Loss: 0.0367 Acc: 0.9953
Val Loss: 0.0031 Acc: 1.0000
running time epoch: 0m 2s

Epoch 8/14
----------
Train Loss: 0.0379 Acc: 0.9907
Val Loss: 0.0067 Acc: 1.0000
running time epoch: 0m 2s

Epoch 9/14
----------
Train Loss: 0.0213 Acc: 1.0000
Val Loss: 0.0031 Acc: 1.0000
running time epoch: 0m 2s

Epoch 10/14
-------